# Project 22 — Distributed Training (DDP / FSDP / TP / PP, all-reduce, scaling efficiency)

```
synthetic MLP + fixed seeds (CPU, gloo-safe: NO live process group; single-proc emulation)
  -> param/byte accounting per strategy: DDP replicate | FSDP shard | TP shard | PP stage
  -> ring all-reduce simulator (scatter-reduce + allgather) VERIFIED vs torch.sum
  -> DDP-equivalence proof: mean(micro-grads) == large-batch grad (<1e-5)
  -> calibrated probes: memcpy BW (GB/s) + MLP step time (ms) on THIS machine
  -> analytic comm model: t = 2(N-1)/N * bytes/BW + N*lat (PCIe vs NVLink)
  -> scaling table: eff = T1/(N*T_N); artifacts: results/dist.csv + scale.csv + dist.png
```

Goal: learn what each parallelism strategy replicates vs shards vs exchanges per step, and why
efficiency falls with N (comm grows) but rises with bandwidth and model size (compute grows).
Objectives: write ring bytes `2(N-1)/N*B` + efficiency `T1/(N*T_N)` + FSDP/TP per-rank bytes from memory.
GPU note: ranks are EMULATED on CPU (gloo spawn hangs notebooks); rerun cell-3's formulas with
`BW=900e9` for NVLink-class numbers, or launch `torchrun --nproc_per_node=2` on DGX Spark for real NCCL.


In [ ]:
import os # need os to make results/ and check cpu count
import random # need random to seed the stdlib RNG
import time # need time to time memcpy + optimizer steps
import csv # need csv to write dist.csv + scale.csv
import numpy as np # need numpy for the sweep tables + means
import torch # need torch for the MLP, grads, and ring-sim tensors
import matplotlib # need matplotlib for the headless figure
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for efficiency curves + comm bars
SEED = 22 # fixed seed: same numbers on every rerun
random.seed(SEED) # seed stdlib RNG
np.random.seed(SEED) # seed numpy RNG
torch.manual_seed(SEED) # seed torch RNG
torch.set_num_threads(os.cpu_count() or 4) # use all CPUs for the timing probes
DEV = torch.device('cpu') # CPU-only: distributed groups are emulated, never spawned
os.makedirs('results', exist_ok=True) # make results/ for CSVs + PNG
print('torch', torch.__version__, '| dist:', torch.distributed.is_available()) # confirm build supports dist APIs

In [ ]:
# ---- model: tiny MLP (stand-in) + exact param/byte accounting per strategy ----
DIN, HID, DOUT = 512, 512, 10 # input width, hidden width, class count
model = torch.nn.Sequential( # 2-layer MLP: the "model" whose bytes we shard
    torch.nn.Linear(DIN, HID), # layer 1: 512x512 + bias
    torch.nn.ReLU(), # nonlinearity between the two matmats
    torch.nn.Linear(HID, DOUT), # layer 2: 512x10 + bias
) # end of tiny MLP definition
P = sum(p.numel() for p in model.parameters()) # exact param count (no guessing)
BFP32 = P * 4 # fp32 bytes of one full replica
BFP16 = P * 2 # bf16/fp16 bytes of one full replica
print(f'params P={P} fp32={BFP32/1e6:.3f}MB fp16={BFP16/1e6:.3f}MB') # show the scale we account from
def ring_bytes(num_bytes, n): # ring all-reduce traffic PER RANK in bytes
    return 2.0 * (n - 1) / n * num_bytes if n > 1 else 0.0 # 2(N-1)/N * B; N=1 sends nothing
def strat_row(name, n, per_rank, comm): # one accounting row for dist.csv
    return {'strategy': name, 'world': n, 'per_rank_MB': round(per_rank / 1e6, 4), # replicate-vs-shard memory
            'comm_MB_step': round(comm / 1e6, 4)} # ring traffic per step at fp32
P100 = 100e6 # reference LLM scale: 100M params for the strategy table
rows = [] # dist.csv rows (memory + traffic per strategy)
rows.append(strat_row('DDP', 8, P100 * 4, ring_bytes(P100 * 4, 8))) # DDP: full replica + grad all-reduce
rows.append(strat_row('FSDP/ZeRO-3', 8, P100 * 4 / 8, 1.5 * ring_bytes(P100 * 4, 8))) # FSDP: 1/N shard, ~1.5x comm
rows.append(strat_row('TP=8', 8, P100 * 4 / 8, 2 * P100 * 4 / 8)) # TP: shard, 2 all-reduces/layer approx
rows.append(strat_row('PP=4', 4, P100 * 4 / 4, P100 * 4 / 4 / 8)) # PP: stage shard, only p2p activations
rows.append(strat_row('DDP', 2, P100 * 4, ring_bytes(P100 * 4, 2))) # DDP N=2: the 1-vs-2 anchor row
for r in rows: # echo the table to the log
    print(r) # one strategy row per line

In [ ]:
# ---- ring simulator: scatter-reduce + allgather, verified against torch.sum ----
def ring_allreduce(vecs): # vecs: list of N rank tensors (same shape); returns list of N sums
    n = len(vecs) # world size N
    chunks = [list(torch.chunk(v, n)) for v in vecs] # split each rank vector into N chunks
    red = [torch.zeros_like(chunks[0][j]) for j in range(n)] # reduced chunks, one owner each
    for j in range(n): # chunk j is owned by rank j after the ring
        acc = torch.zeros_like(chunks[0][j]) # accumulator for chunk j
        for r in range(n): # sum rank r chunk j (the ring would circulate this)
            acc = acc + chunks[r][j] # scatter-reduce step (addition is the op)
        red[j] = acc # owner j now holds the reduced chunk
    out = [] # per-rank outputs (allgather gives everyone everything)
    for r in range(n): # each rank gathers all reduced chunks
        out.append(torch.cat(red)) # concat in chunk order = full sum vector
    return out # N identical sum vectors
N, D = 4, 1024 # 4 emulated ranks, 1K-float vectors
vecs = [torch.randn(D) for _ in range(N)] # one random vector per rank (seeded above)
outs = ring_allreduce(vecs) # run the ring simulator
truth = sum(vecs) # ground truth: plain elementwise sum
for o in outs: # every rank must hold the same sum
    assert torch.max(torch.abs(o - truth)).item() < 1e-5 # ring == torch.sum contract
print('ring ok: max err', max(torch.max(torch.abs(o - truth)).item() for o in outs)) # proof line
# ---- probes: memcpy BW (GB/s) + single-step compute time (ms) on THIS cpu ----
probe = torch.ones(16_000_000, dtype=torch.float32) # 16M fp32 = 64MB probe tensor
t0 = time.perf_counter() # start the copy timer
for _ in range(6): # repeat copies for a stable mean
    probe = probe.clone() # one 64MB local copy (ceiling bandwidth, not link BW)
dt = (time.perf_counter() - t0) / 6 # seconds per 64MB copy
MEM_BW = 64e6 / dt # local copy bandwidth in bytes/s (upper bound; links are slower)
print(f'memcpy {MEM_BW/1e9:.2f} GB/s ({dt*1e3:.2f} ms per 64MB)') # report the ceiling
X = torch.randn(256, DIN) # calibration batch: 256 x 512
Y = torch.randint(0, DOUT, (256,)) # random labels for the step probe
opt = torch.optim.SGD(model.parameters(), lr=0.01) # plain SGD: timing only
crit = torch.nn.CrossEntropyLoss() # mean-reduction CE loss
ts = [] # per-step times
for _ in range(20): # 20 timed steps (first 2 are warmup, dropped below)
    t0 = time.perf_counter() # start step timer
    opt.zero_grad() # clear grads (part of every real step)
    loss = crit(model(X), Y) # forward on the 256-batch
    loss.backward() # backward: builds grads
    opt.step() # SGD update
    ts.append((time.perf_counter() - t0) * 1e3) # ms per step
T1 = float(np.median(ts[2:])) # median step ms, warmup dropped (robust center)
print(f'step median {T1:.2f} ms (n=20, warmup dropped)') # the T1 anchor for efficiency math

In [ ]:
# ---- DDP-equivalence: mean(micro-grads) == large-batch grad (this IS what DDP averages) ----
m = torch.nn.Sequential(torch.nn.Linear(DIN, HID), torch.nn.ReLU(), torch.nn.Linear(HID, DOUT)) # fresh MLP copy
m.load_state_dict(model.state_dict()) # identical init so grads are comparable
Xa, Xb = X[:128].clone(), X[128:].clone() # split batch into two micro-batches (2 emulated ranks)
Ya, Yb = Y[:128].clone(), Y[128:].clone() # matching label halves
m.zero_grad() # clear before the large-batch reference
crit(m(X), Y).backward() # backward on all 256 (mean over 256)
g_full = [p.grad.clone() for p in m.parameters()] # stash reference grads
m.zero_grad() # clear before micro-batch A
crit(m(Xa), Ya).backward() # backward on half A (mean over 128)
gA = [p.grad.clone() for p in m.parameters()] # stash micro grads A
m.zero_grad() # clear before micro-batch B
crit(m(Xb), Yb).backward() # backward on half B (mean over 128)
gB = [p.grad.clone() for p in m.parameters()] # stash micro grads B
errs = [(0.5 * (a + b) - f).abs().max().item() for a, b, f in zip(gA, gB, g_full)] # mean(micros) vs full
print('ddp-equiv max err per tensor:', [f'{e:.2e}' for e in errs]) # expect ~1e-7 (fp32 reassoc only)
assert max(errs) < 1e-5 # DDP averaging == large-batch contract (equal halves)
# ---- FSDP/TP/PP memory + extra-cost math at 100M scale (analytic, exact formulas) ----
f_fsdp = (P100 * 4 / 8) / 1e6 # FSDP/ZeRO-3 per-rank MB: full bytes / N
f_ddp = (P100 * 4) / 1e6 # DDP per-rank MB: full replica (optimizer states excluded both sides)
print(f'FSDP per-rank {f_fsdp:.1f}MB vs DDP {f_ddp:.1f}MB (8x less)') # shard dividend
assert f_fsdp < f_ddp / 7.9 # shard check: must be ~N x smaller
m_stages, p_pp = 8, 4 # 8 micro-batches over a 4-stage pipe
bubble = (p_pp - 1) / m_stages # pipeline bubble fraction (idle drain/fill)
print(f'PP bubble (p=4,m=8): {bubble:.3f}') # idle share of the pipe
assert abs(bubble - 0.375) < 1e-9 # (4-1)/8 contract

In [ ]:
# ---- scaling sweep: strong-scaling efficiency from measured T1 + analytic comm ----
LAT = 5e-6 # 5 us hop latency (NVLink-class; PCIe ~1us but BW dominates here)
def t_comm(num_bytes, n, bw): # analytic all-reduce time in ms
    return (ring_bytes(num_bytes, n) / bw + n * LAT) * 1e3 # bytes/BW + latency term, in ms
BWS = [('PCIe-32GB/s', 32e9), ('NVLink-900GB/s', 900e9)] # two link classes to compare
NS = [1, 2, 4, 8] # world sizes: 1-vs-N scaling
per_param_ms = T1 / P # ms of step per param (linear model: bigger model, more compute)
srows = [] # scale.csv rows
for pname, Ptar in [('tiny-0.3M', float(P)), ('mid-100M', 100e6), ('big-1B', 1e9)]: # three model scales
    Tc = per_param_ms * Ptar # compute ms at this scale (global batch fixed)
    for blabel, bw in BWS: # for each link class
        for n in NS: # for each world size
            tc = t_comm(Ptar * 4, n, bw) # grad all-reduce ms (fp32 DDP)
            tN = Tc / n + tc # no-overlap model: compute shards, comm adds (pessimistic, honest)
            eff = (Tc) / (n * tN) if n > 1 else 1.0 # eff = T1/(N*T_N); N=1 is 1 by definition
            srows.append({'model': pname, 'link': blabel, 'N': n, # config cols
                          't_comm_ms': round(tc, 3), 't_step_ms': round(tN, 3), # timing cols
                          'eff': round(eff, 4), 'speedup': round(eff * n, 4)}) # efficiency cols
for r in srows: # echo sweep to the log
    print(r) # one (model, link, N) row per line
e12 = [r for r in srows if r['model'] == 'mid-100M' and r['link'] == 'NVLink-900GB/s'] # 100M+NVLink slice
assert e12[0]['eff'] == 1.0 # N=1 efficiency is 1 by definition
assert e12[1]['eff'] > e12[2]['eff'] > e12[3]['eff'] # eff falls with N (comm grows, compute shards)
eP = [r for r in srows if r['model'] == 'mid-100M' and r['N'] == 8] # same N, both links
assert eP[1]['eff'] > eP[0]['eff'] # NVLink beats PCIe at fixed N (BW is the lever)
with open('results/dist.csv', 'w', newline='') as f: # write the strategy table
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())) # header from row keys
    w.writeheader() # write header
    w.writerows(rows) # write 5 strategy rows
with open('results/scale.csv', 'w', newline='') as f: # write the scaling sweep
    w = csv.DictWriter(f, fieldnames=list(srows[0].keys())) # header from row keys
    w.writeheader() # write header
    w.writerows(srows) # write 24 sweep rows
fig, ax = plt.subplots(1, 2, figsize=(11, 4)) # two panels: curves + bars
for blabel, mk in [('PCIe-32GB/s', 'o'), ('NVLink-900GB/s', 's')]: # one curve per link (100M model)
    sl = [r for r in srows if r['model'] == 'mid-100M' and r['link'] == blabel] # slice per link
    ax[0].plot([r['N'] for r in sl], [r['eff'] for r in sl], marker=mk, label=blabel) # eff vs N
ax[0].set_xlabel('world size N') # x label
ax[0].set_ylabel('efficiency T1/(N*T_N)') # y label
ax[0].set_title('Strong-scaling eff (100M, DDP)') # panel title
ax[0].legend() # link-class legend
ax[0].grid(True, alpha=0.3) # light grid
labs = [r['strategy'] for r in rows] # strategy names for bars
ax[1].bar(labs, [r['comm_MB_step'] for r in rows]) # comm MB per step per strategy
ax[1].set_ylabel('MB / step / rank') # y label
ax[1].set_title('Per-step traffic (100M, fp32)') # panel title
plt.xticks(rotation=15) # tilt long strategy names
plt.tight_layout() # avoid label clipping
plt.savefig('results/dist.png', dpi=100) # save the figure
print('wrote results/dist.csv results/scale.csv results/dist.png') # artifact confirmation

Cleanup / next steps:
- Artifacts: `results/dist.csv` (strategy memory/traffic), `results/scale.csv` (N × link × size sweep),
  `results/dist.png` (efficiency curves + traffic bars).
- Rerun on DGX Spark: replace `BW` with measured NCCL `all_reduce` perf (`nccl-tests`), keep `T1` probe;
  ring-bytes and efficiency formulas are unchanged (only inputs get real).
- Next: P23 puts a model behind a production API (auth → ratelimit → queue → batch → cache → metrics),
  and P24 wires vision + tracking + this scaling math into the sports-AI capstone.
